# Overview: the FFT Filtering chain layer and its relation to the core API

GridR exposes FFT filtering through two layers.

**Core layer**: `fft_filtering` works on in-memory NumPy arrays. The caller owns the
buffers and pays for holding the whole image in RAM. The module wraps the
`scipy.signal` convolutions and adds what tiled raster processing needs: a production
window, per-side boundary policies, and a decimation fused into the output slicing.

**Chain layer**: `fft_filtering_chain` works on rasterio datasets opened from disk. Its
entry point, `fft_filtering_oa_strip_chain`, cuts the production window into horizontal
strips, reads and writes only what each strip needs, and stitches the strips back
together with the overlap-add method.

This guide states which core features the chain exposes, how it reuses the core
internally, and how each core argument maps to its chain counterpart.

**What you'll learn**

- The conceptual difference between the *core* and *chain* layers
- A complete `core kwarg ↔ chain kwarg` correspondence table
- Which core features the chain restricts, and the role of striping in those restrictions

## The chain layer wraps the core layer

`fft_filtering_chain` does **not** reimplement filtering. For each strip it calls
`fft_array_filter` on the rows it has just read, with the boundary policy that strip
deserves. The chain is responsible for everything around that call:

- Cutting the production window, and not the raster, into strips of `strip_size` rows,
  merging the last two when the last one comes up short.
- Deriving the read window of each strip: the strip extended by the kernel margins at
  the window edges, clipped to the raster, and not extended at the internal cuts.
- Reading those rows from the input dataset through rasterio, into a reused buffer.
- Forcing `zoom=1` and `out_mode="full"` on the per-strip call, so that the partial sums
  are available at full resolution.
- Replacing `boundary` by `"none"` at the internal seams, the outer sides keeping the
  requested policy.
- Summing the partial contributions carried across each seam in the overlap-add buffer.
- Decimating at write time from the global index of the block, then writing that block
  with the optional `round_out` and `binary` conversions.

## Core `↔` Chain argument correspondence

The table below maps every core argument to its chain counterpart. *Forwarded* means the
value reaches the per-strip core call unchanged. Some arguments are instead
*reinterpreted*, because striping changes what they have to mean. *Chain-only* arguments
govern the I/O loop and have no core equivalent.

Two core arguments are not exposed at all: `axes`, since the chain is restricted to 2-D
rasters, and `plan`, which the chain builds for itself.

### Filtering parameters

| Core kwarg | Chain kwarg | Notes |
|---|---|---|
| `kernel` | `kernel` | Forwarded. Even-sized axes are zero-padded to an odd size, as in the core, so that the kernel has a well-defined centre. |
| `win` | `win` | Reinterpreted: the window, and not the raster, is what gets cut into strips. Reads are extended by the kernel margin at the window edges only, using real neighbours wherever they exist. `None` means the whole raster. |
| `boundary` | `boundary` | Reinterpreted per strip: the requested policy is kept on the outer sides of the window and replaced by `"none"` at the internal cuts. As in the core, a policy only applies where the margin leaves the raster, so a window sitting in the interior is filtered identically under every policy. `"wrap"` on the row axis is rejected as soon as the window is actually striped. |
| `out_mode` | `out_mode` | Not forwarded: each strip is computed with `out_mode="full"`, and the requested region is selected at write time. `"valid"` raises `NotImplementedError`. |
| `zoom` | `zoom` | Reinterpreted: the per-strip calls run at `zoom=1`, because overlap-add must sum contributions before any subsampling. The decimation is applied when writing each block. |
| `decimation` | `decimation` | Reinterpreted along with `zoom`: it selects the phase offset, counted from the global index of each block so that the phase survives from one strip to the next. |
| `method` | `method` | Forwarded. |
| `dtype` | `dtype` | Resolved once from the input dataset and the kernel when `None`, then used both for the per-strip calls and for the overlap-add buffer. Pinning it to `float32` halves the memory of the reconstruction. |

### Input source data

| Core kwarg | Chain kwarg | Notes |
|---|---|---|
| `arr` (NumPy) | `ds_in` + `band` | The chain reads the rows each strip needs and forwards that slice. |

### Outputs

| Core kwarg | Chain kwarg | Notes |
|---|---|---|
| return: `FilterResult` | `ds_out` | The chain writes each block to disk and returns `0`; no `FilterResult` is built. `ds_out` must already be open at the decimated output shape, which `fft_array_filter_output_shape` predicts without reading a pixel. The chain raises `ValueError` otherwise, instead of writing a truncated raster. |

### Chain-only: I/O loop

| Chain kwarg | Purpose |
|---|---|
| `round_out` | Boolean option to round the output to the nearest integer. |
| `binary` | Boolean option to binarize the output. |
| `binary_threshold` | In case the `binary` option is activated, all values greater or equal to `binary_threshold` are set to 1, 0 otherwise. |
| `strip_size` | Maximum strip height (in output rows by default). `0` disables stripping. |
| `logger` | Optional `logging.Logger` for debug tracing of the strip / tile loop. |


### Chain-only: I/O loop

| Chain kwarg | Purpose |
|---|---|
| `band` | Band read in the input dataset. |
| `strip_size` | Target strip height, in rows of the production window. Defaults to 512. `0` processes the window in a single call, and the chain falls back to that on its own when the window is too short, or the kernel too tall, for striping to pay off. |
| `round_out` | Rounds the written values to the nearest integer. Defaults to `True`. |
| `binary` | Writes `1` where `abs(value) >= binary_threshold`, `0` elsewhere. Takes precedence over `round_out`. |
| `binary_threshold` | Threshold used by `binary`. Defaults to `1e-3`. |
| `logger` | Optional `logging.Logger` for debug tracing of the strip loop. |

## What this means in practice

Coming from the core tutorials, moving a workflow to the chain layer takes three steps:
replace the NumPy array by an input dataset and a band index, open the output dataset at
the shape returned by `fft_array_filter_output_shape`, and set `strip_size` according to
the available memory. The chain-only conversions `round_out` and `binary` apply when the
output is an integer raster or a mask.

The chain also restricts two options. `out_mode="valid"` raises `NotImplementedError`,
whatever the strip size. A `"wrap"` boundary on the row axis is rejected only when
the window actually ends up striped: the top margin of the first strip is periodic with
the bottom of the window, which a strip cannot see. Passing `strip_size=0` processes the
window in a single call and wraps correctly.
Every other parameter keeps the meaning and the default it has in the core API.

The reconstruction performs a different sequence of floating-point operations than a
single call, with different FFT sizes and partial sums. The chain output therefore
matches the monolithic one to within floating-point tolerance, and not bit for bit.

```{warning}
FFT convolution is global: a single `NaN` contaminates the whole strip it belongs to, and
propagates to the neighbouring strips through the overlap-add seams. Nodata must be
masked or filled before filtering.
```